In [6]:
import requests
from bs4 import BeautifulSoup
import json
import random
import pandas as pd

# Base

In [7]:
import requests
from bs4 import BeautifulSoup
import json
import random

class UserAgentRotator:
    _COMMON_UA_URL = "https://www.useragents.me/"

    _STATIC_UA_POOL = [
        "Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1; SV1; .NET CLR 1.1.4322)",
        "Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko",
        "Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1)",

        # Chrome
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 11_6_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",

        #Firefox
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 12_4) Gecko/20100101 Firefox/114.0",
        "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:113.0) Gecko/20100101 Firefox/113.0",
    ]
    def __init__(self):
        self.update()


    def _fetch_ua(self):

        r = requests.get(self._COMMON_UA_URL)
        soup = BeautifulSoup(r.text, "html.parser")

        ua_div = soup.find("div", attrs={"id": "most-common-desktop-useragents-json-csv"})
        ua = ua_div.find("textarea", class_="form-control")
        ua_list = json.loads(ua.text.strip())
        ua_weight = [entry["pct"] for entry in ua_list]
        ua_list = [entry["ua"] for entry in ua_list]

        return ua_list, ua_weight
    

    def update(self):
        """
        Fetch COMMON UA URL and load UA list to memory
        """
        self.common_ua_list, self.common_ua_weight = self._fetch_ua()

    
    def rotate(self, static_common=[30, 70]):
        use_static_pool = random.choices([True, False], weights=static_common, k=1)[0]
        if use_static_pool:
            return random.choice(self._STATIC_UA_POOL)
        else:
            return random.choices(self.common_ua_list, weights=self.common_ua_weight, k=1)[0]


In [8]:
from abc import ABC, abstractmethod
import logging
import os

class BaseCrawler(ABC):

    def __init__(self, log_level="INFO"):
        self.ua_rotator = UserAgentRotator()
        # self.proxy_manager = ProxyManager()
        self.logger = logging.getLogger(self.__class__.__name__)
        self.setup_logger(log_level)

    
    @abstractmethod
    def crawl(self):
        raise NotImplementedError(f"{self.__class__.__name__} must have crawl method")       
    

    def setup_logger(self, log_level="INFO"):
        log_level = getattr(logging, log_level, logging.INFO)
        self.logger.setLevel(logging.DEBUG)

        console_handler = logging.StreamHandler()
        console_handler.setLevel(log_level)

        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        console_handler.setFormatter(formatter)

        if not self.logger.handlers:
            self.logger.addHandler(console_handler)
            self.logger.propagate = False # prevent duplicate log
        


## Traffic img

In [63]:
import pandas as pd
import json
import os
import re
import requests

class TrafficCrawler(BaseCrawler):
    _BASE_URL = "https://giaothong.hochiminhcity.gov.vn"

    def __init__(self, log_level="INFO"):
        super().__init__(log_level)
        self._cam_image_url = os.path.join(self._BASE_URL, "render/ImageHandler.ashx")
        self._cam_info_url = os.path.join(self._BASE_URL, "ajaxpro/VDMS.Web.Library.AJAX.FolderAjax,VDMS.Web.Library.ashx")
    

    def get_cam_info(self, page: int = 1, limit: int = -1) -> list[dict]:
        with requests.Session() as session:
            payload = {
                "path": "/root/vdms/tangthu/data/layerdata/camera",
                "isInTree": True,
                "searchKey": "",
                "layer": ["CAMERA"],
                "detail": True,
                "page": page,
                "limit": limit,
                "filterQuery": ["Publish:true AND CamStatus:UP"],
                "sortby": {"SortInfo": [
                        {
                            "Field": "ModifiedDate",
                            "Direction": 1
                        }
                    ]},
                "returnFields": ["DisplayName", "CamId", "Disctrict"]
            }

            headers = {
                "Accept": "*/*",
                "Accept-Language": "en-US,en;q=0.9,vi;q=0.8",
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
                "Content-Type": "application/json; charset=UTF-8",
                "Origin": "https://giaothong.hochiminhcity.gov.vn",
                "Pragma": "no-cache",
                "Referer": "https://giaothong.hochiminhcity.gov.vn/",
                "sec-ch-ua-mobile": "?0",
                # "sec-ch-ua-platform": "Windows",
                "Sec-Fetch-Dest": "empty",
                "Sec-Fetch-Mode": "cors",
                "Sec-Fetch-Site": "same-origin",
                "User-Agent": self.ua_rotator.rotate(),
                "X-AjaxPro-Method": "SearchQuery"
            }

            with requests.Session() as session:
                _ = session.get(self._BASE_URL, headers={"User-Agent": self.ua_rotator.rotate()}) # warmup to get cookies
                response = session.request("POST", self._cam_info_url, json=payload, headers=headers)
                # with open("t.txt", "w") as f:
                #     f.write(response.text)
                data = self._extract_cam_info(response.text)
                return data

    
    def _extract_cam_info(self, text) -> list[dict]:
        pattern = r'\[\{"__type":"VDMS\.Sense\.Helper\.Model\.FileProperty, VDMS\.Sense\.Helper\.Model.*?"Format":null\}\]'

        result = []
        matches = re.findall(pattern, text)
        for match in matches:
            json_data = json.loads(match)
            template = {data['Name']: data['Value'] for data in json_data}
            result.append(template)
        
        # Lat Long extract
        pattern = r'\[(.*?new Ajax\.Web\.DataTable\(\[\["GeoId","System\.Object"\],\["Shape","System\.Object"\](?:,\["[^"]+","System\.Object"\])*?\],\[\["([0-9a-f-]{36})","(POINT\(\d+\.\d+ \d+\.\d+\))"(?:,null)*?(?:,"[^"]*?")?\]\]\).*?)\]'
        entries = re.finditer(pattern, text)

        coordinates = {}
        for entry_match in entries:
            entry_content = entry_match.group(1)
            # Extract CamId (third field in the entry, assuming quoted string)
            camid_pattern = r'"[^"]+",null,"([0-9a-f]{24})"'
            camid_match = re.search(camid_pattern, entry_content)
            if camid_match:
                camid = camid_match.group(1)
                # Extract GeoId and Shape from the nested DataTable
                shape = entry_match.group(3)
                shape = shape.replace("POINT(", "").replace(")", "").split(" ") 
                coordinates.update({
                    camid: shape
                })
        
        final = [{
            **cam, 
            "longitude": coordinates.get(cam['CamId'], (None, None))[0],
            "latitude": coordinates.get(cam['CamId'], (None, None))[1]} 
            for cam in result]

        return final
    
    
    def crawl(self, id: str) -> bytes:
        headers = {
            "User-Agent": self.ua_rotator.rotate(),
            # "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8"
        }
        params = {
            "id": id
        }
        response = requests.get(self._cam_image_url, headers=headers, params=params)
        
        # if response.status_code == 200:
        return response.content

In [65]:
CAMERA_DISTRICT_MAPPING = {'662b811d1afb9c00172dcc1d': 'Quận 1',
 '662b85bf1afb9c00172dd149': 'Quận 1',
 '5deb576d1dc17d7c5515ad14': 'Quận 1',
 '662b81721afb9c00172dcc44': 'Quận 1',
 '662b80721afb9c00172dcb28': 'Quận 1',
 '662b857b1afb9c00172dd106': 'Quận 1',
 '5deb576d1dc17d7c5515ad1e': 'Quận 1',
 '662b7d8a1afb9c00172dc71f': 'Quận 1',
 '5deb576d1dc17d7c5515ad06': 'Quận 1',
 '5deb576d1dc17d7c5515ad0c': 'Quận 1',
 '662b7f251afb9c00172dc8bc': 'Quận 1',
 '662b85481afb9c00172dd0f1': 'Quận 1',
 '662b7f9f1afb9c00172dca50': 'Quận 1',
 '662b85031afb9c00172dd0dc': 'Quận 1',
 '662b862a1afb9c00172dd1ff': 'Quận 1',
 '58af994abd82540010390c37': 'Quận 1',
 '5deb576d1dc17d7c5515ad13': 'Quận 1',
 '662b843d1afb9c00172dd02d': 'Quận 1',
 '662b81a31afb9c00172dcc65': 'Quận 1',
 '662b81eb1afb9c00172dcc85': 'Quận 1',
 '5deb576d1dc17d7c5515ad19': 'Quận 1',
 '662b80b91afb9c00172dcb5b': 'Quận 1',
 '5deb576d1dc17d7c5515ad16': 'Quận 1',
 '662b85f51afb9c00172dd1c2': 'Quận 1',
 '662b84771afb9c00172dd076': 'Quận 1',
 '5deb576d1dc17d7c5515ad03': 'Quận 1',
 '6318283cc9eae60017a19f0c': 'Quận 1',
 '6318287ec9eae60017a19f36': 'Quận 1',
 '649da77ea6068200171a6dd4': 'Quận 1',
 '649da72ca6068200171a6dbb': 'Quận 1',
 '5deb576d1dc17d7c5515ad0b': 'Quận 1',
 '63ae7727bfd3d90017e8f14a': 'Quận 4',
 '5deb576d1dc17d7c5515ad1c': 'Quận 4',
 '63ae77bfbfd3d90017e8f18f': 'Quận 4',
 '63b6617ebfd3d90017eaa50b': 'Quận 4',
 '63b661a3bfd3d90017eaa520': 'Quận 4',
 '63ae777cbfd3d90017e8f177': 'Quận 4',
 '63ae7759bfd3d90017e8f162': 'Quận 4',
 '63ae7893bfd3d90017e8f1e1': 'Quận 4',
 '63ae76ddbfd3d90017e8f11b': 'Quận 4',
 '63ae76afbfd3d90017e8f106': 'Quận 4',
 '63ae768dbfd3d90017e8f0f1': 'Quận 4',
 '63ae7669bfd3d90017e8f0d9': 'Quận 4',
 '662b4d781afb9c00172d8571': 'Quận 5',
 '5deb576d1dc17d7c5515ad22': 'Quận 5',
 '662b4ecb1afb9c00172d8692': 'Quận 5',
 '66b1c311779f740018674083': 'Quận 5',
 '662b4de41afb9c00172d85c5': 'Quận 5',
 '66b1c1f2779f740018673f0d': 'Quận 5',
 '5deb576d1dc17d7c5515ad20': 'Quận 5',
 '5b632a79fd4edb0019c7dc0f': 'Quận 5',
 '63b3c274bfd3d90017e9ab93': 'Quận 5',
 '5822f23aedeb6c0012a2d6a8': 'Quận 5',
 '5b728aafca0577001163ff7e': 'Quận 5',
 '5b632b60fd4edb0019c7dc12': 'Quận 5',
 '5b0b7aba0e517b00119fd800': 'Quận 5',
 '5b0b7bbe0e517b00119fd806': 'Quận 5',
 '5b6005b6fd4edb0019c7db25': 'Quận 5',
 '5b6329fdfd4edb0019c7dc0b': 'Quận 5',
 '5d8cd3b7766c880017188942': 'Quận 5',
 '5d8cd49f766c880017188944': 'Quận 5',
 '5d8cd1f9766c880017188938': 'Quận 5',
 '66b1c1bf779f740018673ef2': 'Quận 5',
 '66b1c158779f740018673eb4': 'Quận 5',
 '662b4e201afb9c00172d85f9': 'Quận 5',
 '662b4e581afb9c00172d862f': 'Quận 5',
 '662b4efc1afb9c00172d86bc': 'Quận 5',
 '5deb576d1dc17d7c5515acf3': 'Quận 10',
 '6623e7526f998a001b252407': 'Quận 10',
 '5deb576d1dc17d7c5515acf5': 'Quận 10',
 '63ae7966bfd3d90017e8f240': 'Quận 10',
 '5deb576d1dc17d7c5515ad23': 'Quận 10',
 '6623e6b86f998a001b2523b8': 'Quận 10',
 '6623e5d66f998a001b25235a': 'Quận 10',
 '66b1c34d779f74001867409e': 'Quận 10',
 '6623e7076f998a001b2523ea': 'Quận 10',
 '631955e7c9eae60017a1c30a': 'Quận 10',
 '63ae7a74bfd3d90017e8f2c7': 'Quận 10',
 '63ae7a50bfd3d90017e8f2b2': 'Quận 10',
 '63ae7be0bfd3d90017e8f3a8': 'Quận 10',
 '63ae7c12bfd3d90017e8f3c0': 'Quận 10',
 '63ae7af4bfd3d90017e8f32c': 'Quận 10',
 '63ae7b3cbfd3d90017e8f34d': 'Quận 10',
 '63ae7a26bfd3d90017e8f29a': 'Quận 10',
 '63ae7a9cbfd3d90017e8f303': 'Quận 3',
 '66b1c370779f7400186740b3': 'Quận 10',
 '66b1c398779f7400186740e3': 'Quận 10',
 '5deb576d1dc17d7c5515ad10': 'Quận 3',
 '5deb576d1dc17d7c5515acfb': 'Quận 3',
 '5deb576d1dc17d7c5515ad17': 'Quận 3',
 '5deb576d1dc17d7c5515ad02': 'Quận 3',
 '6623e3ea6f998a001b2522ae': 'Quận 3',
 '5deb576d1dc17d7c5515acfd': 'Quận 3',
 '5deb576d1dc17d7c5515acf2': 'Quận 3',
 '5deb576d1dc17d7c5515ad04': 'Quận 3',
 '5deb576d1dc17d7c5515acf9': 'Quận 3',
 '6623e31e6f998a001b252250': 'Quận 3',
 '5deb576d1dc17d7c5515acfc': 'Quận 3',
 '662b83381afb9c00172dcf88': 'Quận 3',
 '6623e5776f998a001b252337': 'Quận 3',
 '662b84c11afb9c00172dd0b5': 'Quận 3',
 '5deb576d1dc17d7c5515ad11': 'Quận 3',
 '662b830e1afb9c00172dcf50': 'Quận 3',
 '5deb576d1dc17d7c5515acfe': 'Quận 3',
 '6623e43e6f998a001b2522cb': 'Quận 3',
 '6623e5066f998a001b252317': 'Quận 3',
 '5deb576d1dc17d7c5515ad18': 'Quận 3',
 '6623e3a26f998a001b252291': 'Quận 3',
 '6623e4b06f998a001b2522f1': 'Quận 3',
 '662b7ce71afb9c00172dc676': 'Quận 3',
 '5deb576d1dc17d7c5515ad0f': 'Quận 3',
 '5a823d555058170011f6eaa2': 'Quận 3',
 '5ad0621c98d8fc001102e268': 'Quận 3',
 '5deb576d1dc17d7c5515ad0e': 'Quận 3',
 '63ae73cebfd3d90017e8f00d': 'Quận 3',
 '63ae75debfd3d90017e8f082': 'Quận 3',
 '63ae75f9bfd3d90017e8f097': 'Quận 3',
 '5deb576d1dc17d7c5515ad01': 'Quận 3',
 '5deb576d1dc17d7c5515acf8': 'Quận 3',
 '5deb576d1dc17d7c5515ad15': 'Quận 3',
 '6623df636f998a001b251e92': 'Quận 3',
 '6623e2e16f998a001b252233': 'Quận 3',
 '63195512c9eae60017a1c279': 'Quận 3',
 '63195556c9eae60017a1c2ba': 'Quận 3',
 '63ae75a3bfd3d90017e8f051': 'Quận 3',
 '662b80051afb9c00172dcaf6': 'Quận 3',
 '5d9de3c2766c880017188cb3': 'Huyện Nhà Bè',
 '58d8ec0adc195800111e042b': 'Huyện Nhà Bè',
 '62da3e840637a10017d7073d': 'Quận Thủ Đức',
 '6623f1046f998a001b2527db': 'Quận Tân Phú',
 '66b1c3fa779f740018674146': 'Quận Bình Thạnh',
 '662b58791afb9c00172d9107': 'Quận Tân Bình',
 '662b57711afb9c00172d90a7': 'Quận Tân Bình',
 '662b50571afb9c00172d87df': 'Huyện Bình Chánh',
 '6623eec96f998a001b25273b': 'Quận Gò Vấp',
 '662a89ae1afb9c00172d26ad': 'Huyện Bình Chánh',
 '662a8a661afb9c00172d2762': 'Huyện Bình Chánh',
 '662a896e1afb9c00172d2674': 'Huyện Bình Chánh',
 '5deb576d1dc17d7c5515ad05': 'Quận Bình Thạnh',
 '662b5adc1afb9c00172d925b': 'Quận Tân Phú',
 '662b571d1afb9c00172d9083': 'Quận Tân Bình',
 '662a88521afb9c00172d2592': 'Quận Bình Tân',
 '6623ed396f998a001b2526b8': 'Quận Gò Vấp',
 '6623eb466f998a001b252632': 'Quận 12',
 '6623f0246f998a001b252797': 'Quận Tân Phú',
 '662b59291afb9c00172d912b': 'Quận 3',
 '662a87df1afb9c00172d2522': 'Huyện Bình Chánh',
 '662b4faa1afb9c00172d875e': 'Quận 6',
 '6623ea416f998a001b25260a': 'Quận Phú Nhuận',
 '662a881a1afb9c00172d2559': 'Quận Bình Tân',
 '662a8b9e1afb9c00172d284d': 'Huyện Bình Chánh',
 '662b56c51afb9c00172d9071': 'Quận Bình Thạnh',
 '6623ef776f998a001b252768': 'Huyện Hóc Môn',
 '6623e8406f998a001b252465': 'Quận Phú Nhuận',
 '662b4f411afb9c00172d86fc': 'Quận 6',
 '6623e7b76f998a001b25242d': 'Quận Bình Thạnh',
 '662b51761afb9c00172d88ef': 'Huyện Bình Chánh',
 '662b50841afb9c00172d880d': 'Quận 8',
 '6623f44f6f998a001b2528aa': 'Quận 9',
 '662b558c1afb9c00172d8ed2': 'Quận Gò Vấp',
 '662b50c11afb9c00172d8843': 'Quận 6',
 '5deb576d1dc17d7c5515ad09': 'Quận Tân Bình',
 '6623f0576f998a001b2527ac': 'Quận Tân Phú',
 '6623ed9b6f998a001b2526cd': 'Quận Gò Vấp',
 '662b55111afb9c00172d8e2e': 'Quận Gò Vấp',
 '6623ecc16f998a001b25269e': 'Quận Gò Vấp',
 '63b66089bfd3d90017eaa4bd': 'Quận Bình Thạnh',
 '597bf277faa7ea0011c7c29f': 'Quận 9',
 '6623f14b6f998a001b2527f0': 'Quận Tân Phú',
 '66b1c190779f740018673ed4': 'Quận 5',
 '662b52131afb9c00172d8a23': 'Quận Bình Tân',
 '6623f4836f998a001b2528bf': 'Quận 9',
 '63b54a48bfd3d90017ea7850': 'Quận 9',
 '5ad0644698d8fc001102e26b': 'Quận 9',
 '63b54a70bfd3d90017ea7862': 'Quận 9',
 '5ad064b498d8fc001102e26f': 'Quận 9',
 '662a8c381afb9c00172d28b6': 'Quận 8',
 '665861d864f11e00173c73a9': 'Quận 7',
 '662b5c981afb9c00172d94c0': 'Quận Tân Phú',
 '662a861e1afb9c00172d23ad': 'Quận 7',
 '6623f3436f998a001b252863': 'Quận Thủ Đức',
 '662a8cfe1afb9c00172d296e': 'Quận 7',
 '662a88a01afb9c00172d25d1': 'Huyện Bình Chánh',
 '662b4f7e1afb9c00172d872e': 'Quận 6',
 '662b5b481afb9c00172d92a8': 'Quận Tân Bình',
 '5deb576d1dc17d7c5515ad08': 'Quận Tân Bình',
 '59ca30fd02eb490011a0a406': 'Quận Bình Tân',
 '5a6085688576340017d06684': 'Quận Bình Tân',
 '5a6085188576340017d06682': 'Quận Bình Tân',
 '59ca321b02eb490011a0a40d': 'Huyện Bình Chánh',
 '59ca31d602eb490011a0a40b': 'Quận Bình Tân',
 '5a824dc05058170011f6eab2': 'Quận Gò Vấp',
 '5a6084208576340017d0667e': 'Huyện Bình Chánh',
 '5a6088218576340017d06693': 'Quận Bình Tân',
 '5a60839c8576340017d0667c': 'Quận Bình Tân',
 '5a6082698576340017d06678': 'Quận Bình Tân',
 '59ca329d02eb490011a0a410': 'Quận Bình Tân',
 '5a6084898576340017d06680': 'Huyện Bình Chánh',
 '59ca2d9d02eb490011a0a3f0': 'Quận 12',
 '59ca317602eb490011a0a408': 'Quận Bình Tân',
 '5a607fa38576340017d06671': 'Quận Bình Tân',
 '5a6086e88576340017d0668a': 'Quận Bình Tân',
 '5a607f078576340017d0666f': 'Huyện Hóc Môn',
 '6623ee176f998a001b25270c': 'Quận Gò Vấp',
 '5a6087858576340017d0668e': 'Quận Bình Tân',
 '662b5bc31afb9c00172d92e2': 'Quận Tân Phú',
 '5a6085fb8576340017d06686': 'Quận Bình Tân',
 '59ca308302eb490011a0a403': 'Huyện Bình Chánh',
 '59ca301902eb490011a0a400': 'Quận Bình Tân',
 '66b1c22f779f740018673f6e': 'Quận 6',
 '662b50261afb9c00172d87b2': 'Quận 8',
 '5a6066608576340017d06617': 'Quận Gò Vấp',
 '6792efe48c5ed4001b27f336': 'Quận 9',
 '59d3414302eb490011a0a610': 'Quận 9',
 '595dc2f03dcfc400106f2896': 'Quận 12',
 '5a8279865058170011f6eaef': 'Huyện Bình Chánh',
 '5d8cdc9d766c880017188970': 'Quận 11',
 '6623e9e96f998a001b2525ce': 'Quận Phú Nhuận',
 '5deb576d1dc17d7c5515ad0a': 'Quận Tân Bình',
 '5deb576d1dc17d7c5515ad07': 'Quận Phú Nhuận',
 '662a8b061afb9c00172d27d7': 'Quận 7',
 '5a10c79f02eb490011a0b0dd': 'Quận 2',
 '5b87c386ca057700116404a6': 'Quận Thủ Đức',
 '5a6069238576340017d0661c': 'Quận Bình Thạnh',
 '595dd4f43dcfc400106f28ab': 'Quận 12',
 '595dcab53dcfc400106f289d': 'Quận 12',
 '595d9b3b3dcfc400106f287e': 'Quận 12',
 '595dd7693dcfc400106f28b0': 'Quận 12',
 '595d92013dcfc400106f2877': 'Quận Thủ Đức',
 '595dd3a63dcfc400106f28a7': 'Quận 12',
 '595f874d3dcfc400106f28ec': 'Quận 12',
 '595f86c43dcfc400106f28ea': 'Quận 12',
 '595ddc123dcfc400106f28ba': 'Quận 12',
 '595f8d813dcfc400106f28f2': 'Quận 12',
 '595f8d233dcfc400106f28f0': 'Quận 12',
 '595ddb493dcfc400106f28b6': 'Quận 12',
 '595dd9ac3dcfc400106f28b4': 'Quận 12',
 '595dd4133dcfc400106f28a9': 'Quận 12',
 '63ae7c73bfd3d90017e8f3ed': 'Quận 10',
 '5deb576d1dc17d7c5515acfa': 'Quận 3',
 '5deb576d1dc17d7c5515acf4': 'Quận 10',
 '662a86a11afb9c00172d2410': 'Quận 8',
 '662b5be91afb9c00172d939b': 'Quận Tân Phú',
 '65e054fb6b18080018db6632': 'Quận 1',
 '662b5a9c1afb9c00172d9240': 'Quận Tân Bình',
 '662b80e81afb9c00172dcbec': 'Quận 3',
 '66b1c4a2779f74001867418c': 'Quận 3',
 '6623efc26f998a001b25277f': 'Huyện Hóc Môn',
 '662b52a31afb9c00172d8b01': 'Quận Gò Vấp',
 '662b872d1afb9c00172dd36a': 'Quận 3',
 '662b82da1afb9c00172dce94': 'Quận 1',
 '662a87641afb9c00172d24b0': 'Huyện Bình Chánh',
 '662b7d0c1afb9c00172dc6a6': 'Quận 1',
 '6623e7f06f998a001b25244a': 'Quận Bình Thạnh',
 '662a8e121afb9c00172d2a3f': 'Quận 7',
 '662a89f51afb9c00172d26ef': 'Huyện Bình Chánh',
 '6623f1996f998a001b252805': 'Quận 2',
 '662a87a41afb9c00172d24e9': 'Huyện Bình Chánh',
 '662a8ddd1afb9c00172d2a0f': 'Quận 7',
 '662b5b271afb9c00172d9296': 'Quận Tân Bình',
 '5d9de49d766c880017188cb9': 'Quận 9',
 '5a82628e5058170011f6eadb': 'Huyện Bình Chánh',
 '59d34ce302eb490011a0a616': 'Quận 9',
 '5a824c905058170011f6eab0': 'Quận 11',
 '662b50e51afb9c00172d886a': 'Quận 6',
 '63ae7829bfd3d90017e8f1ac': 'Quận 4',
 '63b66051bfd3d90017eaa4a3': 'Quận Bình Thạnh',
 '6818901a6dfb4b0018f9058a': 'Quận Bình Tân',
 '68188fe36dfb4b0018f90520': 'Huyện Bình Chánh',
 '68188cbf6dfb4b0018f900e9': 'Huyện Bình Chánh',
 '68188c876dfb4b0018f9007b': 'Huyện Bình Chánh',
 '68188c236dfb4b0018f8ff8f': 'Huyện Bình Chánh',
 '681889a96dfb4b0018f8f67e': 'Huyện Bình Chánh',
 '63b54898bfd3d90017ea77ae': 'Quận 9',
 '5b8b2323ca057700116405d0': 'Quận Tân Bình',
 '662b54bb1afb9c00172d8dbb': 'Quận Gò Vấp',
 '5deb576d1dc17d7c5515ad1f': 'Quận 5',
 '63ae7cfcbfd3d90017e8f422': 'Quận Tân Bình',
 '63b664d2bfd3d90017eaaa0f': 'Huyện Bình Chánh',
 '5a8254b05058170011f6eac3': 'Quận Bình Thạnh',
 '6792f1008c5ed4001b27f41f': 'Huyện Hóc Môn',
 '5d8cd653766c88001718894c': 'Quận Thủ Đức',
 '662b87551afb9c00172dd43a': 'Quận 3',
 '5d8cda6a766c880017188960': 'Quận Thủ Đức',
 '63b54938bfd3d90017ea77f6': 'Quận 9',
 '63b54909bfd3d90017ea77e4': 'Quận 9',
 '5aab1f852d266a0017e5afd4': 'Quận 7',
 '662a902a1afb9c00172d2bed': 'Huyện Bình Chánh',
 '662a89211afb9c00172d2636': 'Huyện Bình Chánh',
 '6792f3058c5ed4001b27f58e': 'Huyện Bình Chánh',
 '6792ef4f8c5ed4001b27f2c0': 'Huyện Bình Chánh',
 '662a8cc51afb9c00172d2938': 'Quận 7',
 '662b5afe1afb9c00172d9284': 'Quận Tân Phú',
 '662a8c931afb9c00172d2901': 'Quận 7',
 '6623e88c6f998a001b25248b': 'Quận Gò Vấp',
 '6623e61d6f998a001b252377': 'Quận 3',
 '6623f51a6f998a001b252900': 'Quận Thủ Đức',
 '6623edd26f998a001b2526f7': 'Quận Gò Vấp',
 '66b1c4e7779f7400186741e4': 'Quận Tân Bình',
 '662a8d821afb9c00172d29c8': 'Quận 7',
 '662b867b1afb9c00172dd250': 'Quận 1',
 '65e0556b6b18080018db665e': 'Quận 1',
 '662b51a41afb9c00172d8926': 'Quận Bình Tân',
 '65e0552f6b18080018db6647': 'Quận 1',
 '662b5b8c1afb9c00172d92ca': 'Quận Tân Phú',
 '6623ef2b6f998a001b252753': 'Huyện Hóc Môn',
 '6623ebd76f998a001b25264d': 'Quận 12',
 '6623f5876f998a001b25291a': 'Quận 9',
 '662a8ef41afb9c00172d2af2': 'Quận 7',
 '662b86551afb9c00172dd227': 'Quận 1',
 '6623f4df6f998a001b2528eb': 'Quận Thủ Đức',
 '662b83ff1afb9c00172dcffb': 'Quận 1',
 '662a8f3a1afb9c00172d2b31': 'Quận 7',
 '5deb576d1dc17d7c5515ad21': 'Quận 5',
 '6623e8da6f998a001b2524a6': 'Quận Phú Nhuận',
 '662a8e641afb9c00172d2a7e': 'Quận 7',
 '662b57ec1afb9c00172d90e3': 'Quận Tân Bình',
 '6623e3566f998a001b25226d': 'Quận 3',
 '6623ec376f998a001b252671': 'Quận 12',
 '6623f5b56f998a001b25292f': 'Quận 9',
 '662b54711afb9c00172d8d4f': 'Quận Gò Vấp',
 '662b58bd1afb9c00172d9119': 'Quận 3',
 '6623f3d66f998a001b252890': 'Quận Thủ Đức',
 '662a8eb41afb9c00172d2aba': 'Quận 7',
 '662b57471afb9c00172d9095': 'Quận Tân Bình',
 '66b1c426779f74001867415e': 'Quận Bình Thạnh',
 '662b86c41afb9c00172dd31c': 'Quận 1',
 '6623f36e6f998a001b252878': 'Quận Thủ Đức',
 '662b51201afb9c00172d889a': 'Quận Bình Tân',
 '662b58401afb9c00172d90f5': 'Quận Tân Bình',
 '662b5a401afb9c00172d91fc': 'Quận Tân Bình',
 '662b82761afb9c00172dcda3': 'Quận 1',
 '6623ec826f998a001b252686': 'Quận Gò Vấp',
 '662a873b1afb9c00172d2483': 'Huyện Bình Chánh',
 '662b4e8e1afb9c00172d865c': 'Quận 5',
 '662b5c141afb9c00172d93d7': 'Quận Tân Phú',
 '662b836d1afb9c00172dcfa0': 'Quận 1',
 '631828cac9eae60017a19f50': 'Quận 1',
 '6623e6706f998a001b25239b': 'Quận 5',
 '662a8fd41afb9c00172d2bac': 'Quận 7',
 '6623f1f16f998a001b25281f': 'Quận Thủ Đức',
 '63ae79aabfd3d90017e8f26a': 'Quận Tân Bình',
 '6792f9c88c5ed4001b27fb3d': 'Quận Thủ Đức',
 '6792f03d8c5ed4001b27f378': 'Huyện Bình Chánh',
 '62da3e620637a10017d70720': 'Quận Thủ Đức',
 '62da3e390637a10017d706ff': 'Quận Thủ Đức',
 '5a825ded5058170011f6ead7': 'Huyện Bình Chánh',
 '63b54a9ebfd3d90017ea7911': 'Quận 9',
 '5ad068b198d8fc001102e278': 'Huyện Bình Chánh',
 '5a606dbc8576340017d0662b': 'Quận Bình Thạnh',
 '5a823bd55058170011f6eaa0': 'Quận 1',
 '5d8cd9cb766c88001718895c': 'Quận Thủ Đức',
 '6623ee666f998a001b252726': 'Quận Gò Vấp',
 '662b51d21afb9c00172d89bf': 'Quận Bình Tân',
 '662b82a61afb9c00172dcdfd': 'Quận 1',
 '5deb576d1dc17d7c5515acff': 'Quận 3',
 '5a8278f35058170011f6eaed': 'Huyện Bình Chánh',
 '63b65f64bfd3d90017eaa41f': 'Quận Bình Thạnh',
 '63ae798abfd3d90017e8f255': 'Quận 10',
 '63bd1f48bfd3d90017ec3d19': 'Quận 7',
 '5a82439f5058170011f6eaa9': 'Quận 5',
 '63ae7a08bfd3d90017e8f285': 'Quận 10',
 '5bbc7163ca0577001164127f': 'Huyện Nhà Bè',
 '5a8267fe5058170011f6eae1': 'Huyện Nhà Bè',
 '5a606a0f8576340017d0661e': 'Quận Bình Thạnh',
 '6792f16e8c5ed4001b27f482': 'Quận 9',
 '5a8253bc5058170011f6eac1': 'Quận Bình Thạnh',
 '5a606c758576340017d06626': 'Quận Bình Thạnh',
 '5a6060e08576340017d0660f': 'Quận Gò Vấp',
 '5d9ddec9766c880017188c9c': 'Quận Bình Thạnh',
 '5a8253615058170011f6eabf': 'Quận Bình Thạnh',
 '5a606a958576340017d06621': 'Quận Bình Thạnh',
 '5d9ddf49766c880017188ca0': 'Quận Bình Thạnh',
 '5a8254f25058170011f6eac5': 'Quận Bình Thạnh',
 '5a8257e25058170011f6eacd': 'Quận 9',
 '6792f2248c5ed4001b27f4dc': 'Quận Thủ Đức',
 '5b0e1faacddcc80011ceb449': 'Quận Bình Thạnh',
 '6792f3cf8c5ed4001b27f61e': 'Huyện Củ Chi',
 '5d9dde1f766c880017188c98': 'Quận Bình Thạnh',
 '5b0b835b0e517b00119fd80d': 'Quận Thủ Đức',
 '5a606cd08576340017d06628': 'Quận Bình Thạnh',
 '5d9dddb9766c880017188c96': 'Quận Bình Thạnh',
 '5a825b7d5058170011f6ead4': 'Quận 7',
 '5a8259035058170011f6eacf': 'Quận 7',
 '5a8269c45058170011f6eae4': 'Quận 7',
 '66f126e8538c780017c9362f': 'Quận 5',
 '649da419a6068200171a6c90': 'Quận 5',
 '5a8266105058170011f6eadf': 'Huyện Bình Chánh',
 '5ad06a0d98d8fc001102e27b': 'Huyện Bình Chánh',
 '5ad0679598d8fc001102e274': 'Quận 9',
 '5a8256315058170011f6eac9': 'Quận Bình Thạnh',
 '5a82602c5058170011f6ead9': 'Huyện Bình Chánh',
 '66f1266f538c780017c93579': 'Quận 5',
 '5a8256df5058170011f6eacb': 'Quận Bình Thạnh',
 '5a8255a55058170011f6eac7': 'Quận Bình Thạnh',
 '597bf0b8faa7ea0011c7c293': 'Quận 9',
 '63b54c51bfd3d90017ea7aa0': 'Quận 9',
 '63ae7cd3bfd3d90017e8f408': 'Quận 11',
 '5a824b6c5058170011f6eaab': 'Quận Bình Tân',
 '63b54fe4bfd3d90017ea7ca2': 'Quận 2',
 '63b54f70bfd3d90017ea7c86': 'Quận Bình Thạnh',
 '5d8cd7bb766c880017188952': 'Quận Thủ Đức',
 '5d8cd767766c880017188950': 'Quận Bình Thạnh',
 '63ae763bbfd3d90017e8f0c4': 'Quận 3',
 '63b65fd8bfd3d90017eaa461': 'Quận Bình Thạnh',
 '63b55020bfd3d90017ea7cb7': 'Quận 2',
 '63ae7d29bfd3d90017e8f437': 'Quận 10',
 '63b65f8dbfd3d90017eaa434': 'Quận Bình Thạnh',
 '63bd1e95bfd3d90017ec3cd5': 'Quận 7',
 '63b54968bfd3d90017ea7808': 'Quận 9',
 '63b3c2fbbfd3d90017e9abbf': 'Quận Bình Thạnh',
 '5d8cd326766c88001718893e': 'Quận 6',
 '5d8cdc1d766c88001718896c': 'Quận Tân Bình',
 '5a6065c58576340017d06615': 'Quận Thủ Đức',
 '63b54d5abfd3d90017ea7afe': 'Quận 9',
 '5d8cd614766c88001718894a': 'Quận Bình Thạnh',
 '63b66035bfd3d90017eaa48e': 'Quận Bình Thạnh',
 '63b54d27bfd3d90017ea7ae5': 'Quận 9',
 '63b5503bbfd3d90017ea7ccc': 'Quận 2',
 '5a6060a88576340017d0660d': 'Quận Gò Vấp',
 '5a60604f8576340017d0660b': 'Quận Gò Vấp',
 '63b54dcbbfd3d90017ea7ba8': 'Quận Thủ Đức',
 '63b54a04bfd3d90017ea783e': 'Quận 9',
 '63b3c59fbfd3d90017e9ace8': 'Huyện Bình Chánh',
 '5a824f975058170011f6eab8': 'Quận Gò Vấp',
 '59414c883dcfc400106f260b': 'Quận Tân Bình',
 '5a824ee15058170011f6eab6': 'Quận Gò Vấp',
 '63b65fa9bfd3d90017eaa449': 'Quận Bình Thạnh',
 '63b54bcdbfd3d90017ea7a82': 'Quận Thủ Đức',
 '63b54996bfd3d90017ea781a': 'Quận 9',
 '5d8cd98d766c88001718895a': 'Quận Thủ Đức',
 '63ae7c53bfd3d90017e8f3d8': 'Quận 10',
 '5d8cd542766c880017188948': 'Quận Thủ Đức',
 '649da495a6068200171a6cb6': 'Quận 2',
 '5d9de43b766c880017188cb6': 'Quận 9',
 '5d8cdb9f766c880017188968': 'Quận Tân Bình',
 '63b3c9bfbfd3d90017e9b039': 'Quận Thủ Đức',
 '63b549b8bfd3d90017ea782c': 'Quận 9',
 '63b54c93bfd3d90017ea7ab8': 'Quận Thủ Đức'}

In [66]:
import pandas as pd
tf_crawler = TrafficCrawler()
cam_info = tf_crawler.get_cam_info()
# df = pd.DataFrame(cam_info)
# df.rename(columns={
#     "CamId": "id",
#     "DisplayName": "location",
#     "Disctrict": "district"
# }, inplace=True)
# df['district'] = df['id'].map(CAMERA_DISTRICT_MAPPING)
# df['district'] = df['district'].fillna(df['id'].map(CAMERA_DISTRICT_MAPPING))
cam_info

[{'CamId': '5a606dbc8576340017d0662b',
  'DisplayName': 'Điện Biên Phủ - Nguyễn Hữu Cảnh',
  'longitude': '106.670604944229',
  'latitude': '10.846467192202'},
 {'CamId': '6623e31e6f998a001b252250',
  'DisplayName': 'Kỳ Đồng - Bà Huyện Thanh Quan',
  'longitude': '106.681677103043',
  'latitude': '10.7816671586319'},
 {'CamId': '587654d3b807da0011e33d36',
  'Disctrict': 'Quận 2',
  'DisplayName': 'Võ Chí Công - Đường số 2',
  'longitude': None,
  'latitude': None},
 {'CamId': '63b54aedbfd3d90017ea79c3',
  'DisplayName': 'Võ Chí Công - Liên Phường 2',
  'longitude': '106.791937351227',
  'latitude': '10.8049478837466'},
 {'CamId': '58af8a0bbd82540010390c25',
  'Disctrict': 'Quận 3',
  'DisplayName': 'Nam Kỳ Khởi Nghĩa - Võ Thị Sáu',
  'longitude': None,
  'latitude': None},
 {'CamId': '585b2baec3f96200127dc508',
  'Disctrict': 'Quận 7',
  'DisplayName': 'Nguyễn Văn Linh - Huỳnh Tấn Phát 2',
  'longitude': None,
  'latitude': None},
 {'CamId': '595dd4f43dcfc400106f28ab',
  'DisplayName':

In [29]:
df[df['district'].isna()]

,id,district,location
4,595da2853dcfc400106f2883,NaN,Quốc lộ 1 - Đường 15 (1)
8,595da2dd3dcfc400106f2885,NaN,Quốc lộ 1 - Đường 15 (2)
24,63bd1f21bfd3d90017ec3d04,NaN,Huỳnh Tấn Phát - Phạm Hữu Lầu
29,63ae7938bfd3d90017e8f226,NaN,Quốc Lộ 1- Công ty Pouyen 3 (Cổng)
69,5b5995a3fd4edb0019c7d9ab,NaN,Võ Văn Kiệt - Cầu Chà Và 2 (trên cầu)
72,56de42f611f398ec0c481295,NaN,Võ Văn Kiệt - Cầu Chà Và 1 (dạ cầu)
74,66b1c3c2779f740018674125,NaN,Vòng xoay Mũi Tàu Phú Lâm
75,56de42f611f398ec0c48128c,NaN,Võ Văn Kiệt - Nguyễn Văn Cừ 2
84,5deb576d1dc17d7c5515acf6,NaN,Nút giao Ngã sáu Cộng Hòa
89,5deb576d1dc17d7c5515acf7,NaN,Nút giao Ngã sáu Cộng Hòa


## weather

In [5]:
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import unicodedata
import requests
import os
import re

class WeatherCrawler(BaseCrawler):
    _BASE_URL = "https://www.accuweather.com"

    def __init__(self):
        super().__init__()
        self._url_cities = os.path.join(self._BASE_URL, "vi/browse-locations/asi/vn")

        self._headers = {
            "Referer": "https://www.accuweather.com"
        }

    def crawl(self, url: str):
        result = {}
        soup = BeautifulSoup(requests.get(url, headers=self._headers).text, "html.parser")
        current_weather = soup.find("div", class_="current-weather-card card-module content-module")
        current_time = current_weather.find("p", class_="sub").text
        status = current_weather.find("div", class_="phrase").text # Sunny
        temp_c = current_weather.find("div", class_="display-temp").text
        properties = current_weather.find_all("div", class_="detail-item spaced-content")

        result.update({
            "current_time": current_time,
            "status": status,
            "temp_c": temp_c
        })

        for prop in properties:
            detail = prop.text.split("\n")
            detail = [_ for _ in detail if _ != ""]
            result.update({detail[0]: detail[1]})
        result

    def get_city_list(self) -> list[dict]:
        self._headers.update({"User-Agent": self.ua_rotator.rotate()})
    
        url = os.path.join(self._BASE_URL, "vi/browse-locations/asi/vn")
        headers = {
            "User-Agent": "insomnia/11.0.2",
            "Referer": "https://www.accuweather.com"
        }
        with requests.Session() as s:
            r = s.get(url, headers=headers)

        soup = BeautifulSoup(r.text, "html.parser")
        search_result = [tag for tag in soup.find_all("a", class_="search-result") if tag.get("class") == ["search-result"]]
        cities = [{"city": tag.text, "url": self._BASE_URL + tag["href"]} for tag in search_result]
        result = []
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = [executor.submit(self._get_district, city, headers) for city in cities]
            
            for future in as_completed(futures):
                try:
                    result.extend(future.result())
                except Exception as e:
                    self.logger.error(f"Error: {e}")

        return result

    def _preprocess(self, district: dict):
        # Process name
        d_name = district["district"].lower().replace(" ", "-").replace("quận", "district")
        d_name = unicodedata.normalize('NFD', d_name)
        d_name = ''.join(c for c in d_name if unicodedata.category(c) != 'Mn')
        d_name = d_name.replace('đ', 'd').replace('Đ', 'D')

        # Extract key
        match = re.search(r'key=(\d+)', district["district_url"])
        if match:
            key = match.group(1)
        
        return {"district_url": f"https://www.accuweather.com/en/vn/{d_name}/{key}/current-weather/{key}", "district": district['district']}
    

    def _get_district(self, city, headers):
        with requests.Session() as s:
            response = s.get(city['url'], headers=headers)
            soup = BeautifulSoup(response.text, "html.parser")
            districts = [self._preprocess({"district": tag.text, "district_url": tag.get("href")}) for tag in soup.find_all("a", class_="search-result") if tag.get("class") == ["search-result"]]
            districts = [{**district, **city} for district in districts]
            return districts


In [5]:
crawler = WeatherCrawler()
a = crawler.get_city_list()
a
# df = pd.DataFrame(a)
# df.to_csv("temp_weather.tsv", sep="\t", index=True, header=False)

[{'district_url': 'https://www.accuweather.com/vi/vn/ap-binh-chau/352096/weather-forecast/352096',
  'district': 'Ấp Bình Châu',
  'city': 'Bà Rịa - Vũng Tàu',
  'url': 'https://www.accuweather.com/vi/browse-locations/asi/vn/43'},
 {'district_url': 'https://www.accuweather.com/vi/vn/ba-ria/352095/weather-forecast/352095',
  'district': 'Bà Rịa',
  'city': 'Bà Rịa - Vũng Tàu',
  'url': 'https://www.accuweather.com/vi/browse-locations/asi/vn/43'},
 {'district_url': 'https://www.accuweather.com/vi/vn/chau-thanh/352091/weather-forecast/352091',
  'district': 'Châu Thành',
  'city': 'Bà Rịa - Vũng Tàu',
  'url': 'https://www.accuweather.com/vi/browse-locations/asi/vn/43'},
 {'district_url': 'https://www.accuweather.com/vi/vn/cho-phuoc-hai/352090/weather-forecast/352090',
  'district': 'Chợ Phước Hải',
  'city': 'Bà Rịa - Vũng Tàu',
  'url': 'https://www.accuweather.com/vi/browse-locations/asi/vn/43'},
 {'district_url': 'https://www.accuweather.com/vi/vn/co-ong/428244/weather-forecast/428244

In [19]:
df.to_csv("temp_weather.tsv", sep="\t", index=False, header=True)

In [ ]:
from bs4 import BeautifulSoup
import requests
headers = {
    "User-Agent": "insomnia/11.0.2",
    "Referer": "https://www.accuweather.com"
}
url = "https://www.accuweather.com/en/vn/district-1/3554433/current-weather/3554433"

with requests.Session() as session:
    r = session.get(url, headers=headers)

result = {}
soup = BeautifulSoup(r.text, "html.parser")
current_weather = soup.find("div", class_="current-weather-card card-module content-module")
current_time = current_weather.find("p", class_="sub").text
status = current_weather.find("div", class_="phrase").text # Sunny
temp_c = current_weather.find("div", class_="display-temp").text
properties = current_weather.find_all("div", class_="detail-item spaced-content")

result.update({
    "current_time": current_time,
    "status": status,
    "temp_c": temp_c
})

for prop in properties:
    detail = prop.text.split("\n")
    detail = [_ for _ in detail if _ != ""]
    result.update({detail[0]: detail[1]})
result

In [38]:
result = {}
soup = BeautifulSoup(r.text, "html.parser")
current_weather = soup.find("div", class_="current-weather-card card-module content-module")
current_time = current_weather.find("p", class_="sub").text
status = current_weather.find("div", class_="phrase").text # Sunny
temp_c = current_weather.find("div", class_="display-temp").text
properties = current_weather.find_all("div", class_="detail-item spaced-content")

result.update({
    "current_time": current_time,
    "status": status,
    "temp_c": temp_c
})

for prop in properties:
    detail = prop.text.split("\n")
    detail = [_ for _ in detail if _ != ""]
    result.update({detail[0]: detail[1]})
result

{'current_time': '4:33 PM',
 'status': 'Sunny',
 'temp_c': '35°C\n',
 'RealFeel®': '39°',
 'RealFeel Shade™': '38°',
 'Max UV Index': '3 Moderate',
 'Wind': 'S 9 km/h',
 'Wind Gusts': '9 km/h',
 'Humidity': '46%',
 'Indoor Humidity': '46% (Extremely Humid)',
 'Dew Point': '22° C',
 'Pressure': '↔ 1006 mb',
 'Cloud Cover': '10%',
 'Visibility': '16 km',
 'Cloud Ceiling': '12200 m'}

In [13]:
headers = {
    "User-Agent": "insomnia/11.0.2",
    "Referer": "https://www.accuweather.com"
}
requests.get("https://www.accuweather.com/vi/vn/district-1/3554433/current-weather/3554433", headers=headers).text

'\n\n<!DOCTYPE html>\n<html lang="vi" class="accuweather">\n\n<head>\n\t<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\n\t\n\t<meta charset="utf-8" />\n\t<link rel="canonical" href="https://www.accuweather.com/vi/vn/district-1/3554433/current-weather/3554433" />\n\t<title>Th&#x1EDD;i ti&#x1EBF;t hi&#x1EC7;n t&#x1EA1;i &#x1EDF; Qu&#x1EAD;n 1, H&#x1ED3; Ch&#xED; Minh, Vi&#x1EC7;t Nam | AccuWeather</title>\n\t<meta name="Description" content="H&#xE3;y chu&#x1EA9;n b&#x1ECB; k&#x1EF9; cho ng&#xE0;y. H&#xE3;y ki&#x1EC3;m tra t&#xEC;nh tr&#x1EA1;ng hi&#x1EC7;n t&#x1EA1;i cho Qu&#x1EAD;n 1, H&#x1ED3; Ch&#xED; Minh, Vi&#x1EC7;t Nam cho ng&#xE0;y s&#x1EAF;p t&#x1EDB;i, c&#xF3; th&#xF4;ng tin radar, d&#x1EF1; b&#xE1;o theo gi&#x1EDD; v&#xE0; chi ti&#x1EBF;t &#x111;&#x1EBF;n m&#x1EE9;c theo ph&#xFA;t. ">\n\t<meta name="viewport" content="width=device-width, initial-scale=1.0" />\n\t<meta name="referrer" content="origin">\n\t\n\t\n\n\t<meta property="fb:profile_id" content="AccuWea

In [7]:
MODEL_API = "http://localhost:8000/model/object_counting/predict"
crawler = TrafficCrawler()
img_data = crawler.crawl("595d9b3b3dcfc400106f287e")
files = {'file': ('file.png', img_data, 'image/png')}
message = requests.post(MODEL_API, files=files).json()
message
        

{'image_shape': [288, 512],
 'total': 2,
 'objects': [{'class_object': 'truck',
   'coordinates': [412.47430419921875,
    68.10554504394531,
    496.2538146972656,
    145.9583282470703],
   'confidence': 0.51,
   'class_id': 7,
   'classname': 'truck'},
  {'class_object': 'car',
   'coordinates': [302.49969482421875,
    45.04814529418945,
    323.7963562011719,
    64.5850601196289],
   'confidence': 0.39,
   'class_id': 2,
   'classname': 'car'}]}

In [16]:
data = pd.read_csv("district.csv")
data = {i['id']: i['district'] for i in data.to_dict(orient="records")}
data

{'5d9de3c2766c880017188cb3': 'Huyện Nhà Bè',
 '58d8ec0adc195800111e042b': 'Huyện Nhà Bè',
 '63b54a48bfd3d90017ea7850': 'Quận 9',
 '62da3e840637a10017d7073d': 'Quận Thủ Đức',
 '5ad0644698d8fc001102e26b': 'Quận 9',
 '63b54a70bfd3d90017ea7862': 'Quận 9',
 '5ad064b498d8fc001102e26f': 'Quận 9',
 '6623f1046f998a001b2527db': 'Quận Tân Phú',
 '66b1c3fa779f740018674146': 'Quận Bình Thạnh',
 '662b58791afb9c00172d9107': 'Quận Tân Bình',
 '662b57711afb9c00172d90a7': 'Quận Tân Bình',
 '662b50571afb9c00172d87df': 'Huyện Bình Chánh',
 '6623eec96f998a001b25273b': 'Quận Gò Vấp',
 '662a89ae1afb9c00172d26ad': 'Huyện Bình Chánh',
 '662a8a661afb9c00172d2762': 'Huyện Bình Chánh',
 '662a896e1afb9c00172d2674': 'Huyện Bình Chánh',
 '5deb576d1dc17d7c5515ad05': 'Quận Bình Thạnh',
 '662b5adc1afb9c00172d925b': 'Quận Tân Phú',
 '662b571d1afb9c00172d9083': 'Quận Tân Bình',
 '662a88521afb9c00172d2592': 'Quận Bình Tân',
 '6623ed396f998a001b2526b8': 'Quận Gò Vấp',
 '6623eb466f998a001b252632': 'Quận 12',
 '6623f0246f998